In [ ]:
# On Colab, data lives in Google Drive; locally, next to this notebook
import sys
if 'google.colab' in sys.modules:
  from google.colab import drive
  drive.mount('/content/drive')
  DATA_DIR = '/content/drive/MyDrive/EWN-AI'
else:
  DATA_DIR = '.'

In [ ]:
import copy
import json
import os
import time
import numpy as np
import h5py
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
%matplotlib inline

# Data format

e.g. `[523416,342516],3/c5-d4,3/e1xd2,6/a3-b3,5/e2xd2,2/b5xb4,6/e3-d3,4/a4xb4,2/c1-b2,3/d4-e3,3/b2-a3,6/b3-c2,6/d3-c4,5/a5xb4,6/c4-b5,5/b4-c3,1/a3-a4,3/e3-e2,4/d1xd2,1/e2-e1,1-0`

  1. Blue initial board and red initial board
  2. Moves of the two players in turn, written as `dice/source-destination` (`x` instead of `-` for a capture). Blue starts this example, but in the later games of a match either side may move first
  3. Game result: `1-0` is a blue win, `0-1` is a red win

# Feature map
Same layout as `board_to_FM()` in `EWN/model.cpp`. The game asks the model about one piece at a time, marked in channel 12.

0-5: Positions of red chess number 1 to 6

6-11: Positions of blue chess number 1 to 6

12: Position of the chess to move

13: All is 1, which lets the convolutions find the board edges

14: How many dice faces (0-6) let the chess on this square move

15: How many dice faces (0-6) let the opponent move a chess onto this square next turn

In board: 0 is empty, 1-6 is red, 7-12 is blue. Red moves toward (4, 4) and blue toward (0, 0).

Label: direction of the move relative to the moving side, as `direct` in `Game::update_game_status()`: 0 is horizontal, 1 is vertical, 2 is diagonal. The model outputs one probability per direction, `probs[direction]` of `model_probs()` in `model.cpp`.

In [ ]:
with open(f'{DATA_DIR}/training_data.csv') as f:
  games = f.read().splitlines()

chars = 'edcba'
cols = {k:v for v,k in enumerate(chars)}
blue_init_pos = [(4, 4), (4, 3), (4, 2), (3, 4), (3, 3), (2, 4)]
red_init_pos = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (2, 0)]

In [ ]:
def parse_move(move):
  src, dst = move[2:4], move[5:7]
  return int(src[1])-1, cols[src[0]], int(dst[1])-1, cols[dst[0]]

def dice_movers(board, base):
  # For each dice face, the chess of one side (base 0 is red, 6 is blue) it lets move, as in
  # Game::count_movable_chs(): its own chess, or the nearest lower and higher ones if it was captured
  alive = [k for k in range(1, 7) if (board == base + k).any()]
  for dice in range(1, 7):
    lower = [k for k in alive if k <= dice]
    higher = [k for k in alive if k >= dice]
    yield {lower[-1] if lower else 0, higher[0] if higher else 0} - {0}

def dice_coverage(board):
  # How many dice faces let each chess move
  counts = np.zeros((5, 5), dtype=np.uint8)
  for base in (0, 6):
    for movers in dice_movers(board, base):
      for k in movers:
        counts[board == base + k] += 1
  return counts

def threat_map(board, red_to_move):
  # How many dice faces let the opponent move a chess onto each square next turn
  base, step = (6, -1) if red_to_move else (0, 1)
  counts = np.zeros((5, 5), dtype=np.uint8)
  for movers in dice_movers(board, base):
    reach = np.zeros((5, 5), dtype=bool)
    for k in movers:
      (r, c), = np.argwhere(board == base + k)
      for dr, dc in ((0, step), (step, 0), (step, step)):
        if 0 <= r + dr < 5 and 0 <= c + dc < 5:
          reach[r + dr, c + dc] = True
    counts += reach
  return counts

def prepare_input(board, row, col):
  # (row, col) is the chess to move
  x = np.zeros((16, 5, 5), dtype=np.uint8)
  rows, columns = np.nonzero(board)
  x[board[rows, columns] - 1, rows, columns] = 1
  x[12, row, col] = 1
  x[13] = 1
  x[14] = dice_coverage(board)
  x[15] = threat_map(board, board[row, col] <= 6)
  return x

def prepare_label(src_row, src_col, dst_row, dst_col):
  if src_row == dst_row:
    return 0
  if src_col == dst_col:
    return 1
  return 2

def initial_board(setup):
  board = np.zeros((5, 5), dtype=int)
  for i in range(6):
    row, col = red_init_pos[i]
    board[row, col] = int(setup[1][i])

    row, col = blue_init_pos[i]
    board[row, col] = int(setup[0][i+1]) + 6
  return board

def game_samples(game):
  # Yields (feature map, label, whether the mover won the game) for every move
  fields = game.split(',')
  board, moves = initial_board(fields[:2]), fields[2:-1]
  blue_won = fields[-1] == '1-0'
  for move in moves:
    src_row, src_col, dst_row, dst_col = parse_move(move)
    # Either side may move first in the later games of a match, so the mover is read from the board
    mover_won = (board[src_row, src_col] > 6) == blue_won
    yield prepare_input(board, src_row, src_col), prepare_label(src_row, src_col, dst_row, dst_col), mover_won

    # update board
    board[dst_row, dst_col] = board[src_row, src_col]
    board[src_row, src_col] = 0

# CSV to HDF5
Games are split into training and test sets by game, so positions from the same game never end up in both. Only the original positions are stored; the board symmetries are applied during training.

The HDF5 file caches the feature maps, so it is only rebuilt when it is missing or older than the CSV. Set `REBUILD` to `True` after changing `prepare_input()` or `TEST_RATIO`.

In [ ]:
filename = f'{DATA_DIR}/ewn_train.h5'
TEST_RATIO = 0.15
REBUILD = False

if REBUILD or not os.path.exists(filename) or os.path.getmtime(filename) < os.path.getmtime(f'{DATA_DIR}/training_data.csv'):
  rng = np.random.default_rng(3)
  is_test = rng.random(len(games)) < TEST_RATIO

  with h5py.File(filename, 'w') as hf:
    for split in ('train', 'test'):
      hf.create_dataset(f'{split}/x', shape=(0, 16, 5, 5), maxshape=(None, 16, 5, 5), dtype='u1', chunks=True)
      hf.create_dataset(f'{split}/y', shape=(0,), maxshape=(None,), dtype='u1', chunks=True)
      hf.create_dataset(f'{split}/won', shape=(0,), maxshape=(None,), dtype='bool', chunks=True)

    # Data processing, 5000 games at a time
    for start in range(0, len(games), 5000):
      samples = {'train': [], 'test': []}
      for game, test in zip(games[start:start+5000], is_test[start:start+5000]):
        samples['test' if test else 'train'] += game_samples(game)

      for split, rows in samples.items():
        if not rows:
          continue
        for key, data in zip(('x', 'y', 'won'), map(np.array, zip(*rows))):
          dataset = hf[f'{split}/{key}']
          dataset.resize(dataset.shape[0] + len(data), axis=0)
          dataset[-len(data):] = data
else:
  print(f'{filename} is newer than the CSV, so it is reused')

with h5py.File(filename, 'r') as hf:
  print(f'{len(games)} games, moves per split:', {split: hf[f"{split}/y"].shape[0] for split in ('train', 'test')})

# Data loading
The whole dataset is kept on the device as uint8 and batched by index, which is much faster than a `DataLoader` over nearly a million small samples.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(3)

with h5py.File(filename, 'r') as hf:
  train_data = [torch.from_numpy(hf[f'train/{key}'][:]).to(device) for key in ('x', 'y', 'won')]
  test_data = [torch.from_numpy(hf[f'test/{key}'][:]).to(device) for key in ('x', 'y', 'won')]

def batches(data, batch_size, shuffle=False):
  n = data[0].shape[0]
  order = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
  for i in range(0, n, batch_size):
    x, y, won = (t[order[i:i+batch_size]] for t in data)
    yield x.float(), y.long(), won

# Legal moves and symmetries
`legal_mask` follows `Game::check_in_board()`: a direction is illegal when it would move the chess off the board. Illegal directions are masked out of the loss and the accuracy, and `model.cpp` gives them a probability of 0 in the game.

Two board symmetries keep the rules unchanged, so each training sample is randomly transformed by them:
- Rotating the board 180 degrees and swapping colors turns a red move into the same blue move, because the label is relative to the moving side
- Transposing the board keeps both home corners in place and swaps horizontal and vertical moves

In [ ]:
def legal_mask(x):
  # (N, 3) bool mask of the directions that keep the chess in channel 12 on the board
  piece = x[:, 12]
  index = torch.arange(5, device=x.device, dtype=x.dtype)
  row = (piece.sum(2) * index).sum(1)
  col = (piece.sum(1) * index).sum(1)
  is_red = (x[:, 0:6].sum(1) * piece).sum((1, 2)) > 0
  can_move_col = torch.where(is_red, col < 4, col > 0)
  can_move_row = torch.where(is_red, row < 4, row > 0)
  return torch.stack([can_move_col, can_move_row, can_move_col & can_move_row], dim=1)

SWAP_COLORS = [6, 7, 8, 9, 10, 11, 0, 1, 2, 3, 4, 5, 12, 13, 14, 15]
SWAP_HV = torch.tensor([1, 0, 2], device=device)

def augment(x, y):
  rotate = torch.rand(x.shape[0], device=x.device) < 0.5
  x = torch.where(rotate.view(-1, 1, 1, 1), x[:, SWAP_COLORS].flip(2, 3), x)
  transpose = torch.rand(x.shape[0], device=x.device) < 0.5
  x = torch.where(transpose.view(-1, 1, 1, 1), x.transpose(2, 3), x)
  y = torch.where(transpose, SWAP_HV[y], y)
  return x, y

# Model
A small AlphaZero-style residual network. `export_weights()` writes it for the game, which runs it in `model.cpp` and turns its output into the probabilities of the legal directions.

In [ ]:
class ResidualBlock(nn.Module):
  def __init__(self, channels):
    super().__init__()

    self.net = nn.Sequential(
      nn.Conv2d(channels, channels, 3, padding=1, bias=False),
      nn.BatchNorm2d(channels),
      nn.ReLU(),
      nn.Conv2d(channels, channels, 3, padding=1, bias=False),
      nn.BatchNorm2d(channels)
    )

  def forward(self, x):
    return F.relu(x + self.net(x))

class PolicyNet(nn.Module):
  def __init__(self, blocks=6, channels=64):
    super().__init__()

    self.net = nn.Sequential(
      nn.Conv2d(16, channels, 3, padding=1, bias=False),
      nn.BatchNorm2d(channels),
      nn.ReLU(),
      *[ResidualBlock(channels) for _ in range(blocks)],
      nn.Conv2d(channels, 2, 1, bias=False),
      nn.BatchNorm2d(2),
      nn.ReLU(),
      nn.Flatten(),
      nn.Linear(2*5*5, 3)
    )

  def forward(self, x):
    return self.net(x)

def export_weights(model, path):
  # The game runs the network itself in EWN/model.cpp. The file holds 'EWN1' and the numbers of input channels,
  # channels and residual blocks as int32, then in float32 the weight and bias of every convolution, with the
  # batch norm after it folded in, and of the linear layer
  def folded(conv, bn):
    scale = bn.weight / torch.sqrt(bn.running_var + bn.eps)
    return [conv.weight * scale.view(-1, 1, 1, 1), bn.bias - bn.running_mean * scale]

  with torch.no_grad():
    layers = list(model.net)
    blocks = [layer.net for layer in layers if isinstance(layer, ResidualBlock)]
    tensors = folded(layers[0], layers[1])
    for block in blocks:
      tensors += folded(block[0], block[1]) + folded(block[3], block[4])
    tensors += folded(layers[-5], layers[-4]) + [layers[-1].weight, layers[-1].bias]
  with open(path, 'wb') as f:
    f.write(b'EWN1' + np.array([layers[0].in_channels, layers[0].out_channels, len(blocks)], dtype='<i4').tobytes())
    for t in tensors:
      f.write(t.detach().float().cpu().numpy().astype('<f4').tobytes())

model = PolicyNet().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Model training
Moves of the side that lost the game count `LOSER_WEIGHT` in the loss. Training runs for `EPOCHS` epochs on batches of `BATCH_SIZE`, with a one-cycle learning rate schedule and mixed precision on the GPU; every improvement of the test loss is exported to `model_ewn.bin`. When training ends, `model_ewn.json` records the data, settings and test results of the exported model.

In [ ]:
LOSER_WEIGHT = 0.5  # loss weight of the moves played by the side that lost the game
BATCH_SIZE = 1024
EPOCHS = 20
# Mixed precision runs the network on the GPU's tensor cores; the masking and the loss stay in float32
use_amp = device.type == 'cuda'
torch.backends.cudnn.benchmark = True

def masked_logits(model, x):
  return model(x).float().masked_fill(~legal_mask(x), -1e9)

def train(model, dataset, optimizer, scheduler, scaler):
  model.train()
  # Summed on the device, so the GPU never waits for the CPU within an epoch
  train_loss, train_acc, count = 0, 0, 0
  for data, label, won in batches(dataset, BATCH_SIZE, shuffle=True):
    data, label = augment(data, label)
    with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
      output = masked_logits(model, data)

    weight = torch.where(won, 1.0, LOSER_WEIGHT)
    loss = (F.cross_entropy(output, label, reduction='none') * weight).sum() / weight.sum()

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    train_loss += loss.detach() * len(label)
    train_acc += (output.argmax(1) == label).sum()
    count += len(label)
  return train_loss.item() / count, train_acc.item() / count

def test(model, dataset, batch_size=4096):
  model.eval()
  test_loss, test_acc, count = 0, 0, 0
  with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
    for data, label, _ in batches(dataset, batch_size):
      output = masked_logits(model, data)
      test_loss += F.cross_entropy(output, label, reduction='sum').item()
      test_acc += (output.argmax(1) == label).sum().item()
      count += len(label)
  return test_loss / count, test_acc / count

class Checkpoint:
  # Exports the model whenever the test loss improves, and keeps the best weights and results
  def __init__(self, save_path):
    self.save_path = save_path
    self.val_loss_min = float('inf')
    self.best_state = None
    self.best_results = None

  def update(self, val_loss, model, results):
    if val_loss < self.val_loss_min:
      print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). saving model...')
      export_weights(model, self.save_path)
      self.best_state = copy.deepcopy(model.state_dict())
      self.best_results = results
      self.val_loss_min = val_loss

In [ ]:
# Baselines on the test set, to compare the model's accuracy with
data, label = test_data[0].float(), test_data[1].long()
legal = legal_mask(data)

# The most frequent direction in the training set among the legal ones
frequency = torch.bincount(train_data[1].long(), minlength=3).float()
baseline_accuracy = ((legal * frequency).argmax(1) == label).float().mean().item()
random_legal = 1 / legal.sum(1).float()

print(f'Most frequent legal direction: {baseline_accuracy:.4f}')
print(f'Random legal direction: {random_legal.mean():.4f}')
del data, label, legal

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# Warms up to max_lr and then anneals; stepped once per batch
steps_per_epoch = (train_data[0].shape[0] + BATCH_SIZE - 1) // BATCH_SIZE
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, epochs=EPOCHS, steps_per_epoch=steps_per_epoch)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

train_loss_list, train_acc_list = [], []
test_loss_list, test_acc_list = [], []

checkpoint = Checkpoint(f'{DATA_DIR}/model_ewn.bin')
training_start = time.time()

for epoch in range(EPOCHS):
  epoch_start = time.time()
  train_loss, train_acc = train(model, train_data, optimizer, scheduler, scaler)
  train_loss_list.append(train_loss)
  train_acc_list.append(train_acc)

  test_loss, test_acc = test(model, test_data)
  test_loss_list.append(test_loss)
  test_acc_list.append(test_acc)

  print(f'Epoch {epoch}: train loss: {train_loss:.6f}, train accuracy: {train_acc:.4f}' +
     f', test loss: {test_loss:.6f}, test accuracy: {test_acc:.4f}, {time.time() - epoch_start:.0f}s')
  checkpoint.update(test_loss, model, {'epoch': epoch, 'test_loss': round(test_loss, 6), 'test_accuracy': round(test_acc, 4)})

print(f'Training time: {(time.time() - training_start) / 60:.1f} minutes')
model.load_state_dict(checkpoint.best_state)

# How the exported model was trained, to keep next to it
with open(f'{DATA_DIR}/model_ewn.json', 'w') as f:
  json.dump({
    'exported': time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime()),
    'torch': torch.__version__,
    'games': len(games),
    'train_moves': len(train_data[1]),
    'test_moves': len(test_data[1]),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'loser_weight': LOSER_WEIGHT,
    'best_epoch': checkpoint.best_results['epoch'],
    'test_loss': checkpoint.best_results['test_loss'],
    'test_accuracy': checkpoint.best_results['test_accuracy'],
    'baseline_accuracy': round(baseline_accuracy, 4),
  }, f, indent=2)

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(train_loss_list, color='red', label='train loss')
plt.plot(test_loss_list, color='blue', label='test loss')
plt.legend(loc='upper right')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.xticks(range(len(train_loss_list)))
plt.tight_layout()

plt.subplot(1, 2, 2)
plt.plot(train_acc_list, color='red', label='train acc')
plt.plot(test_acc_list, color='blue', label='test acc')
plt.legend(loc='upper left')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.xticks(range(len(train_loss_list)))
plt.tight_layout()

# Prediction
Copy `model_ewn.bin` and `model_ewn.json` into `EWN/` to use the model in the game.

In [ ]:
model.eval()

board = np.array([[3, 4, 2, 0, 0]
          ,[5, 1, 0, 0, 0]
          ,[6, 0, 0, 0, 12]
          ,[0, 0, 0, 7, 10]
          ,[0, 0, 9, 8, 11]])
# The first move of the example game in "Data format": blue 3 at c5
src_row, src_col, _, _ = parse_move('3/c5-d4')
features = torch.from_numpy(prepare_input(board, src_row, src_col)).unsqueeze(0).to(device, dtype=torch.float)
torch.set_printoptions(precision=4, sci_mode=False)
# Probabilities of horizontal, vertical, diagonal
with torch.no_grad():
  probs = F.softmax(masked_logits(model, features), dim=1)
probs